### **CÉLULA 1: IMPORTAÇÃO DE BIBLIOTECAS**

Nesta célula, importamos todas as bibliotecas necessárias para a análise e construção dos modelos de Machine Learning. Cada biblioteca tem uma função específica:

*   **`pandas` (pd)**: Utilizada para manipulação e análise de dados, especialmente DataFrames.
*   **`numpy` (np)**: Fundamental para operações numéricas e matemáticas de alto desempenho.
*   **`matplotlib.pyplot` (plt)** e **`seaborn` (sns)**: Usadas para visualização de dados e criação de gráficos estatísticos.
*   **`sklearn.model_selection.train_test_split`**: Para dividir os dados em conjuntos de treinamento e teste.
*   **`sklearn.preprocessing.LabelEncoder`** e **`StandardScaler`**: Para transformar variáveis categóricas em numéricas e para normalizar as features — etapa **obrigatória** para SVM e RNA, que são sensíveis à escala dos dados.
*   **`sklearn.svm.SVC`**: O algoritmo **Support Vector Machine** (Classificador).
*   **`sklearn.neural_network.MLPClassifier`**: O algoritmo de **Rede Neural Artificial** (Perceptron Multi-Camadas).
*   **`sklearn.metrics`**: Contém funções para avaliar o desempenho dos modelos (acurácia, precisão, recall, F1-Score e matriz de confusão).
*   **`sklearn.inspection.permutation_importance`**: Usada para calcular a importância das features de forma agnóstica ao modelo.

O `sns.set(style="whitegrid")` configura um estilo padrão para os gráficos.

In [ ]:
# ==============================================================================
# CÉLULA 1: IMPORTAÇÃO DE BIBLIOTECAS
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Bibliotecas de Machine Learning (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)
from sklearn.inspection import permutation_importance

# Configuração de estilo dos gráficos
sns.set(style="whitegrid")

# Import opcional para desbalanceamento
try:
    from imblearn.over_sampling import SMOTE
    smote_disponivel = True
except Exception:
    smote_disponivel = False

print("Bibliotecas importadas com sucesso!")
print(f"SMOTE disponível: {smote_disponivel}")

### **CÉLULA 2: CARREGAMENTO E VISUALIZAÇÃO DOS DADOS**

Esta célula é responsável por carregar o conjunto de dados (`baseMLJurandir.csv`) em um DataFrame do pandas. Após o carregamento, exibimos as primeiras 5 linhas (`df.head()`) para ter uma visão inicial dos dados e confirmamos suas dimensões (`df.shape`).

Um bloco `try-except` é usado para lidar com a situação em que o arquivo não é encontrado, fornecendo uma mensagem de erro útil para o usuário.

In [ ]:
# ==============================================================================
# CÉLULA 2: CARREGAMENTO E VISUALIZAÇÃO DOS DADOS
# ==============================================================================
try:
    df = pd.read_csv('baseMLJurandir.csv')
    print(f"Dataset carregado com sucesso! Dimensões: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print("ERRO: O arquivo 'baseMLJurandir.csv' não foi encontrado.")
    print("Por favor, faça o upload do arquivo na aba de arquivos à esquerda ou verifique o caminho.")

### **CÉLULA 3: PRÉ-PROCESSAMENTO E ENGENHARIA DE FEATURES**

Esta é uma etapa crucial para preparar os dados para o treinamento do modelo. Realizamos diversas transformações:

1.  **Correção de Colunas com Erro de Encoding**: Muitas vezes, caracteres especiais (como `ú` ou `ção`) podem ser carregados incorretamente (ex: `Ãºcleo`, `Ã§Ã£o`). Criamos uma função auxiliar (`corrigir_coluna`) para identificar e renomear essas colunas para seus nomes corretos (`núcleo`, `modulação`), garantindo que o código possa referenciá-las corretamente.

2.  **Criação da Variável Target (`aceito`)**: A coluna original `resultado` é usada para criar uma nova variável binária chamada `aceito`. Se `resultado` for `1`, `aceito` é `1` (indicando que o circuito foi aceito); caso contrário, `aceito` é `0` (indicando bloqueio). Esta será a variável que nossos modelos tentarão prever.

3.  **Remoção da Coluna Original `resultado`**: Após criar `aceito`, a coluna `resultado` se torna redundante e é removida do DataFrame.

4.  **Criação da Feature `slots_usados`**: Duas colunas existentes, `primeiro slot` e `ultimo slot`, são combinadas para criar uma nova e mais informativa feature, `slots_usados`, que representa a quantidade de slots utilizados. As colunas originais são então removidas.

5.  **Tratamento da Variável Categórica `modulação`**: A coluna `modulação` é uma variável categórica. Para que os algoritmos de Machine Learning possam processá-la, ela é convertida em um formato numérico usando `LabelEncoder`.

Ao final, exibimos a estrutura do DataFrame processado e suas colunas para verificar as mudanças.

In [ ]:
# ==============================================================================
# CÉLULA 3: PRÉ-PROCESSAMENTO E ENGENHARIA DE FEATURES
# ==============================================================================

print("Colunas do DataFrame antes do pré-processamento:", df.columns.tolist())

# --- Correção de Colunas com Erro de Encoding ---
def corrigir_coluna(df, prefixo_esperado, sufixos_erro, nome_correto):
    found_col = None
    for col in df.columns:
        if prefixo_esperado in col.lower():
            for sufixo in sufixos_erro:
                if sufixo in col:
                    found_col = col
                    break
            if found_col:
                break

    if found_col and found_col != nome_correto:
        df.rename(columns={found_col: nome_correto}, inplace=True)
        print(f"Coluna '{found_col}' renomeada para '{nome_correto}'.")
    elif found_col and found_col == nome_correto:
        print(f"Coluna '{nome_correto}' já está com o nome correto.")
    else:
        print(f"AVISO: A coluna '{nome_correto}' (ou sua variação com encoding errado) não foi encontrada no DataFrame.")

# Corrigir 'núcleo'
corrigir_coluna(df, 'n', ['Ãºcleo'], 'núcleo')

# Corrigir 'modulação'
corrigir_coluna(df, 'modula', ['Ã§Ã£o', 'ção'], 'modulação')
# --- Fim da Correção de Colunas ---

# 1. Criar variável target binária "aceito"
df['aceito'] = df['resultado'].apply(lambda x: 1 if x == 1 else 0)

# 2. Remover a coluna original "resultado"
df.drop('resultado', axis=1, inplace=True)

# 3. Criar feature "slots_usados" e remover "primeiro slot" e "ultimo slot"
df['slots_usados'] = df['ultimo slot'] - df['primeiro slot'] + 1
df.drop(['primeiro slot', 'ultimo slot'], axis=1, inplace=True)

# 4. Tratar variável categórica "modulação" com LabelEncoder
le = LabelEncoder()
if 'modulação' in df.columns and df['modulação'].dtype == 'object':
    df['modulação'] = le.fit_transform(df['modulação'])
    print("Classes de modulação codificadas:", list(le.classes_))
elif 'modulação' in df.columns:
    print("A coluna 'modulação' já parece ser numérica ou foi convertida, nenhuma alteração feita.")
else:
    print("ERRO: A coluna 'modulação' não foi encontrada no DataFrame após a tentativa de correção.")

# 5. Features derivadas de domínio (interações simples)
if 'slots_usados' in df.columns and 'modulação' in df.columns:
    df['interacao_slots_modulacao'] = df['slots_usados'] * df['modulação']

if 'slots_usados' in df.columns and 'núcleo' in df.columns:
    df['interacao_slots_nucleo'] = df['slots_usados'] * df['núcleo']

print("\n--- Estrutura Final do Dataset ---")
display(df.head())
print(f"Colunas finais: {df.columns.tolist()}")

### **CÉLULA 4: SEPARAÇÃO EM TREINO E TESTE (COM AMOSTRAGEM RÁPIDA)**

Para validar rapidamente se o pipeline está funcionando, usamos uma **fração pequena e reprodutível** do dataset antes de treinar.

Fluxo adotado:

1. Definimos `sample_frac` (ex.: `0.05` = 5% dos dados).
2. Geramos `df_model` por amostragem estratificada via classe alvo (`aceito`), preservando a proporção entre classes.
3. Dividimos em treino/teste com `train_test_split` (70/30).

Quando quiser rodar com o dataset completo, basta definir `sample_frac = 1.0`.

In [ ]:
# ==============================================================================
# CÉLULA 4: SEPARAÇÃO EM TREINO E TESTE
# ==============================================================================

# Ajuste rápido para validação do pipeline
sample_frac = 0.05  # use 1.0 para rodar com 100% da base

# Amostragem estratificada robusta para manter proporção de classes
if sample_frac < 1.0:
    partes = []
    for _, grupo in df.groupby('aceito'):
        n_amostras = max(2, int(round(len(grupo) * sample_frac)))
        n_amostras = min(n_amostras, len(grupo))
        partes.append(grupo.sample(n=n_amostras, random_state=42))
    df_model = pd.concat(partes, axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)
else:
    df_model = df.copy()

# Definir X (features) e y (target)
X = df_model.drop('aceito', axis=1)
y = df_model['aceito']

# Diagnóstico de desbalanceamento
dist_classes = y.value_counts(normalize=True).sort_index()
minority_ratio = dist_classes.min()
imbalanco_alto = minority_ratio < 0.35
usar_class_weight_balanced = imbalanco_alto

print("Distribuição de classes (proporção):")
print(dist_classes)
print(f"Desbalanceamento alto? {imbalanco_alto}")

# Separar: 70% Treino, 30% Teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Tamanho original da base: {df.shape[0]} amostras")
print(f"Tamanho usado no experimento: {df_model.shape[0]} amostras ({sample_frac:.0%})")
print(f"Tamanho do Treino: {X_train.shape[0]} amostras")
print(f"Tamanho do Teste:  {X_test.shape[0]} amostras")

### **CÉLULA 5: NORMALIZAÇÃO DAS FEATURES (StandardScaler)**

Esta etapa é **exclusiva e obrigatória** para os algoritmos SVM e RNA, sendo desnecessária no Random Forest (que é invariante à escala).

#### **Por que escalar os dados?**

*   **SVM**: O algoritmo busca o hiperplano de margem máxima entre as classes. Se as features tiverem escalas muito diferentes (ex: uma feature varia de 0 a 1000 e outra de 0 a 1), a que tem maior magnitude dominará o cálculo da distância, distorcendo completamente o hiperplano ótimo.

*   **RNA (MLPClassifier)**: A Rede Neural usa gradiente descendente para otimizar os pesos. Com features em escalas díspares, o gradiente se torna excessivamente assimétrico, causando convergência lenta ou instável.

#### **Como funciona o `StandardScaler`?**

Ele transforma cada feature para ter **média 0** e **desvio padrão 1** (padronização Z-score):

$$z = \frac{x - \mu}{\sigma}$$

**Regra de ouro**: O scaler é ajustado (`fit`) **apenas nos dados de treino** para evitar *data leakage*. Em seguida, é aplicado (`transform`) tanto no treino quanto no teste.

In [ ]:
# ==============================================================================
# CÉLULA 5: NORMALIZAÇÃO DAS FEATURES (StandardScaler)
# ==============================================================================

scaler = StandardScaler()

# IMPORTANTE: fit() apenas no treino para evitar data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SMOTE apenas no treino (se houver desbalanceamento alto e biblioteca disponível)
usar_smote = imbalanco_alto and smote_disponivel
if usar_smote:
    smote = SMOTE(random_state=42)
    X_train_final, y_train_final = smote.fit_resample(X_train_scaled, y_train)
    print("SMOTE aplicado no conjunto de treino.")
else:
    X_train_final, y_train_final = X_train_scaled, y_train
    if imbalanco_alto and not smote_disponivel:
        print("SMOTE não aplicado: biblioteca imblearn não disponível.")
    else:
        print("SMOTE não necessário para este nível de balanceamento.")

print("Normalização concluída com StandardScaler.")
print(f"Média das features (treino após scaling): {X_train_scaled.mean(axis=0).round(4)}")
print(f"Desvio padrão (treino após scaling):      {X_train_scaled.std(axis=0).round(4)}")
print(f"Treino final para modelos: {X_train_final.shape[0]} amostras")

### **CÉLULA 6: TREINAMENTO DOS MODELOS (CONFIGURAÇÃO OTIMIZADA DE TEMPO)**

Nesta célula, inicializamos e treinamos dois modelos de classificação com foco em **boa performance + menor tempo computacional**:

1.  **Support Vector Machine (SVC)**: Utilizamos o **kernel linear** para reduzir significativamente o custo de treinamento em relação ao RBF em bases maiores.
    *   `C=1.0` — regularização padrão.
    *   `kernel='linear'` — versão mais rápida para treino.
    *   `probability=False` — evita o custo extra de calibração de probabilidade durante o treino.

2.  **RNA — Rede Neural Artificial (MLPClassifier)**: Mantemos uma arquitetura eficiente, porém mais leve para convergir mais rápido.
    *   `hidden_layer_sizes=(64, 32)` — arquitetura menor e mais rápida.
    *   `activation='relu'`, `solver='adam'`.
    *   `max_iter=200` com `early_stopping=True`.

Ambos os modelos são treinados (`.fit()`) usando os dados **normalizados** de treino (`X_train_scaled`, `y_train`).

In [ ]:
# ==============================================================================
# CÉLULA 6: TREINAMENTO DOS MODELOS
# ==============================================================================

# Definir class_weight para SVM conforme diagnóstico de desbalanceamento
svm_class_weight = 'balanced' if usar_class_weight_balanced else None

# 1. Support Vector Machine (SVC com kernel linear - mais rápido)
svm_model = SVC(
    C=0.1,
    kernel='linear',
    probability=False,
    random_state=42
)
svm_model.fit(X_train_final, y_train_final)
print("Modelo SVM (kernel linear) treinado.")
print(f"  class_weight: {svm_class_weight}")
print(f"  Número de vetores de suporte por classe: {svm_model.n_support_}")

# 2. Rede Neural Artificial (MLP - configuração mais leve)
rna_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=200,
    tol=1e-3,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)
rna_model.fit(X_train_final, y_train_final)
print(f"\nModelo RNA (MLP) treinado.")
print(f"  Camadas: {rna_model.n_layers_} | Iterações realizadas: {rna_model.n_iter_}")

### **CÉLULA 7: AVALIAÇÃO DOS MODELOS**

Após treinar os modelos, avaliamos seu desempenho com uma função auxiliar `avaliar_modelo` que calcula métricas e exibe a matriz de confusão.

#### **Métricas de Avaliação:**

*   **Acurácia (Accuracy)**: Proporção de previsões corretas sobre o total.
*   **Precisão (Precision)**: Das previsões positivas, quantas estavam corretas? Minimiza **falsos positivos** (circuito bloqueado previsto como aceito).
*   **Recall (Sensibilidade)**: Dos casos realmente positivos, quantos foram identificados? Minimiza **falsos negativos** (circuito válido previsto como bloqueio). Um **alto Recall** para `aceito=1` significa que o modelo não bloqueia erroneamente circuitos válidos.
*   **F1-Score**: Média harmônica entre Precisão e Recall. Útil em datasets desbalanceados.

#### **Matriz de Confusão:**

Mostra a distribuição de:
*   **VP** — Previu Aceito, era Aceito.
*   **VN** — Previu Bloqueio, era Bloqueio.
*   **FP** — Previu Aceito, era Bloqueio (circuito bloqueado aceito erroneamente).
*   **FN** — Previu Bloqueio, era Aceito (circuito válido bloqueado erroneamente).

O `classification_report` fornece um resumo detalhado por classe.

In [ ]:
# ==============================================================================
# CÉLULA 7: AVALIAÇÃO DOS MODELOS
# ==============================================================================

def avaliar_modelo(modelo, nome_modelo, X_test_s, y_test):
    """Função auxiliar para calcular métricas e plotar matriz de confusão."""
    y_pred = modelo.predict(X_test_s)

    # Métricas
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)

    print(f"--- Resultados: {nome_modelo} ---")
    print(f"Acurácia:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("\nRelatório de Classificação:")
    print(classification_report(y_test, y_pred))

    # Matriz de Confusão
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Matriz de Confusão — {nome_modelo}')
    plt.xlabel('Predito (0=Bloqueio, 1=Aceito)')
    plt.ylabel('Real (0=Bloqueio, 1=Aceito)')
    plt.tight_layout()
    plt.show()

    return [acc, prec, rec, f1]

# Avaliar SVM
metrics_svm = avaliar_modelo(svm_model, "SVM (kernel linear)", X_test_scaled, y_test)

# Avaliar RNA
metrics_rna = avaliar_modelo(rna_model, "RNA (MLP)", X_test_scaled, y_test)

### **CÉLULA 8: COMPARAÇÃO FINAL DOS MODELOS**

Nesta célula, consolidamos as métricas de avaliação de ambos os modelos (SVM e RNA) em um DataFrame para facilitar a comparação lado a lado. Em seguida, visualizamos essas métricas com um gráfico de barras comparativo.

Essa comparação é fundamental para escolher o modelo mais adequado para implantação, levando em conta os objetivos do negócio — por exemplo, se é mais crítico minimizar falsos negativos (circuitos válidos bloqueados) ou falsos positivos (circuitos inválidos aceitos).

In [ ]:
# ==============================================================================
# CÉLULA 8: COMPARAÇÃO FINAL DOS MODELOS
# ==============================================================================

# Criar DataFrame comparativo
df_compare = pd.DataFrame({
    'Métrica':  ['Acurácia', 'Precision', 'Recall', 'F1-Score'],
    'SVM (Linear)': metrics_svm,
    'RNA (MLP)': metrics_rna
})

print("\n--- Comparação Lado a Lado ---")
display(df_compare)

# Gráfico de barras comparativo
df_compare_melted = df_compare.melt(id_vars="Métrica", var_name="Modelo", value_name="Valor")
plt.figure(figsize=(10, 6))
sns.barplot(x="Métrica", y="Valor", hue="Modelo", data=df_compare_melted, palette="magma")
plt.title("Comparação de Desempenho: SVM Linear vs RNA (MLP)")
plt.ylim(0, 1.1)
plt.tight_layout()
plt.show()

### **CÉLULA 9: IMPORTÂNCIA DAS FEATURES + REDUÇÃO DE FEATURES**

Como SVM e RNA não possuem `feature_importances_` nativo, usamos **Permutation Importance**.

Além de visualizar a importância, esta célula também:

1. Identifica features com impacto quase nulo.
2. Remove essas features (com base na importância média entre SVM e RNA).
3. Retreina os modelos com o conjunto reduzido.
4. Reavalia as métricas para comparar desempenho vs simplicidade.

A lógica é manter o modelo mais leve sem perda relevante de desempenho.

In [ ]:
# ==============================================================================
# CÉLULA 9: IMPORTÂNCIA DAS FEATURES + REDUÇÃO DE FEATURES
# ==============================================================================

feature_names = X.columns.tolist()

# Subamostragem do teste para acelerar o cálculo
rng = np.random.default_rng(42)
max_amostras_importancia = 1000
if len(X_test_scaled) > max_amostras_importancia:
    idx = rng.choice(len(X_test_scaled), size=max_amostras_importancia, replace=False)
    X_test_imp = X_test_scaled[idx]
    y_test_imp = y_test.iloc[idx].values
else:
    X_test_imp = X_test_scaled
    y_test_imp = y_test.values

def calcular_importancia(modelo, X_eval, y_eval):
    return permutation_importance(
        modelo, X_eval, y_eval,
        n_repeats=5,
        random_state=42,
        scoring='accuracy'
    )

def plotar_importancia(result, nome_modelo, feature_names, color):
    sorted_idx = result.importances_mean.argsort()[::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(
        range(len(feature_names)),
        result.importances_mean[sorted_idx],
        xerr=result.importances_std[sorted_idx],
        align='center',
        color=color,
        alpha=0.85
    )
    plt.yticks(range(len(feature_names)), [feature_names[i] for i in sorted_idx])
    plt.gca().invert_yaxis()
    plt.xlabel("Queda média na acurácia (maior = mais importante)")
    plt.title(f"Permutation Importance — {nome_modelo}")
    plt.tight_layout()
    plt.show()

print(f"Amostras usadas na importance: {len(X_test_imp)}")

print("Calculando Permutation Importance para o SVM...")
imp_svm = calcular_importancia(svm_model, X_test_imp, y_test_imp)
plotar_importancia(imp_svm, "SVM (kernel linear)", feature_names, color="steelblue")

print("\nCalculando Permutation Importance para a RNA (MLP)...")
imp_rna = calcular_importancia(rna_model, X_test_imp, y_test_imp)
plotar_importancia(imp_rna, "RNA (MLP)", feature_names, color="coral")

# Consolidação da importância para seleção de features
imp_media = (imp_svm.importances_mean + imp_rna.importances_mean) / 2
limiar_importancia = 0.001
selected_idx = np.where(imp_media > limiar_importancia)[0]

# Garante pelo menos 3 features no cenário extremo
if len(selected_idx) < 3:
    selected_idx = np.argsort(imp_media)[-3:]

selected_features = [feature_names[i] for i in selected_idx]
removed_features = [feature_names[i] for i in range(len(feature_names)) if i not in selected_idx]

print("\n--- Seleção por importância ---")
print(f"Limiar usado: {limiar_importancia}")
print(f"Features mantidas ({len(selected_features)}): {selected_features}")
print(f"Features removidas ({len(removed_features)}): {removed_features}")

# Reavaliar modelos com features reduzidas
X_train_red = X_train_final[:, selected_idx]
X_test_red = X_test_scaled[:, selected_idx]

def avaliar_sem_plot(modelo, X_t, y_t):
    pred = modelo.predict(X_t)
    return {
        'Acurácia': accuracy_score(y_t, pred),
        'Precision': precision_score(y_t, pred, zero_division=0),
        'Recall': recall_score(y_t, pred, zero_division=0),
        'F1-Score': f1_score(y_t, pred, zero_division=0)
    }

svm_red = SVC(
    C=0.1,
    kernel='linear',
    probability=False,
    class_weight=svm_class_weight,
    random_state=42
)
svm_red.fit(X_train_red, y_train_final)

rna_red = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=200,
    tol=1e-3,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)
rna_red.fit(X_train_red, y_train_final)

metricas_originais = pd.DataFrame({
    'Modelo': ['SVM Original', 'RNA Original'],
    **{
        'Acurácia': [metrics_svm[0], metrics_rna[0]],
        'Precision': [metrics_svm[1], metrics_rna[1]],
        'Recall': [metrics_svm[2], metrics_rna[2]],
        'F1-Score': [metrics_svm[3], metrics_rna[3]]
    }
})

met_svm_red = avaliar_sem_plot(svm_red, X_test_red, y_test)
met_rna_red = avaliar_sem_plot(rna_red, X_test_red, y_test)

metricas_reduzidas = pd.DataFrame([
    {'Modelo': 'SVM Reduzido', **met_svm_red},
    {'Modelo': 'RNA Reduzido', **met_rna_red}
])

print("\n--- Comparação: modelos originais vs reduzidos ---")
display(pd.concat([metricas_originais, metricas_reduzidas], ignore_index=True))

### **CÉLULA 10: AJUSTE DE LIMIAR ORIENTADO AO NEGÓCIO (PRECISION-RECALL)**

Nesta etapa, o objetivo não é apenas maximizar métricas globais, mas reduzir o custo de erro do negócio:

- **FN (Falso Negativo)**: circuito válido bloqueado (erro mais crítico).
- **FP (Falso Positivo)**: circuito indevido aceito.

Usamos a curva Precision-Recall para varrer limiares e escolher o ponto com menor custo total:

$$Custo = (FN \times custo_{FN}) + (FP \times custo_{FP})$$

Assim, a decisão final passa a refletir o impacto operacional real dos erros.

In [ ]:
# ==============================================================================
# CÉLULA 10: AJUSTE DE LIMIAR ORIENTADO AO NEGÓCIO (PRECISION-RECALL)
# ==============================================================================

# Custos de negócio (ajuste conforme operação real)
custo_fn = 5  # bloquear circuito válido
custo_fp = 1  # aceitar circuito indevido

def custo_erro(y_true, y_pred, c_fn=5, c_fp=1):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    custo_total = fn * c_fn + fp * c_fp
    return {'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp, 'Custo': custo_total}

def melhor_limiar_por_custo(y_true, scores, limiar_base, c_fn=5, c_fp=1):
    thresholds = np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 120)))

    melhor = None
    for t in thresholds:
        pred = (scores >= t).astype(int)
        custo = custo_erro(y_true, pred, c_fn, c_fp)
        if (melhor is None) or (custo['Custo'] < melhor['Custo']):
            melhor = {'threshold': float(t), **custo}

    pred_base = (scores >= limiar_base).astype(int)
    base = {'threshold': float(limiar_base), **custo_erro(y_true, pred_base, c_fn, c_fp)}
    return base, melhor

# Scores para limiar
score_svm = svm_model.decision_function(X_test_scaled)           # limiar base natural = 0.0
score_rna = rna_model.predict_proba(X_test_scaled)[:, 1]         # limiar base natural = 0.5

# Curvas PR
prec_svm, rec_svm, _ = precision_recall_curve(y_test, score_svm)
prec_rna, rec_rna, _ = precision_recall_curve(y_test, score_rna)
ap_svm = average_precision_score(y_test, score_svm)
ap_rna = average_precision_score(y_test, score_rna)

# Escolha de limiar por custo
svm_base, svm_otimo = melhor_limiar_por_custo(y_test.values, score_svm, limiar_base=0.0, c_fn=custo_fn, c_fp=custo_fp)
rna_base, rna_otimo = melhor_limiar_por_custo(y_test.values, score_rna, limiar_base=0.5, c_fn=custo_fn, c_fp=custo_fp)

# Gráfico PR
plt.figure(figsize=(8, 6))
plt.plot(rec_svm, prec_svm, label=f"SVM (AP={ap_svm:.3f})")
plt.plot(rec_rna, prec_rna, label=f"RNA (AP={ap_rna:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# Tabela de decisão por custo
df_custos = pd.DataFrame([
    {'Modelo': 'SVM', 'Cenário': 'Base', **svm_base},
    {'Modelo': 'SVM', 'Cenário': 'Ótimo por custo', **svm_otimo},
    {'Modelo': 'RNA', 'Cenário': 'Base', **rna_base},
    {'Modelo': 'RNA', 'Cenário': 'Ótimo por custo', **rna_otimo},
])

print(f"Custos adotados: FN={custo_fn}, FP={custo_fp}")
print("\n--- Comparação por custo de erro (FN vs FP) ---")
display(df_custos[['Modelo', 'Cenário', 'threshold', 'TN', 'FP', 'FN', 'TP', 'Custo']])